In [36]:
import os,math,warnings,math,pickle,random

from mpl_toolkits.mplot3d.proj3d import transform
from numpy.array_api import trunc
warnings.filterwarnings('ignore')
from collections import defaultdict

import pandas as pd
import numpy as np
from tqdm import tqdm,tqdm_notebook

# 用于向量检索的库
import faiss

from sklearn.preprocessing import MinMaxScaler,LabelEncoder
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow.keras as keras
import copy

In [37]:
metric_recall=True # 是否要调试，如果不调试就是用全量数据集

# 1.读取数据

In [38]:
def get_all_click_sample(sample_nums=10000):
    '''debug模式，从训练集中抽取一部分数据调试代码'''
    all_click=pd.read_csv('./data/train_click_log.csv')
    all_user_ids=all_click.user_id.unique()
    sample_user_ids=np.random.choice(all_user_ids,size=sample_nums,replace=False)
    all_click=all_click[all_click['user_id'].isin(sample_user_ids)]
    all_click=all_click.drop_duplicates(['user_id','click_article_id','click_timestamp'])
    return all_click

def get_all_click_df(offline):
    '''读取点击数据，分为线上和线下，线下用训练集数据，线上用训练集+测试集'''
    if offline:
        all_click=pd.read_csv('./data/train_click_log.csv')
    else:
        trn_click=pd.read_csv('./data/train_click_log.csv')
        tst_click=pd.read_csv('./data/testA_click_log.csv')
        all_click=pd.concat([trn_click,tst_click]).reset_index(drop=True)
    all_click=all_click.drop_duplicates(['user_id','click_article_id','click_timestamp'])
    return all_click

In [39]:
def get_item_info_df():
    '''读取文章的基本属性'''
    item_info_df=pd.read_csv('./data/articles.csv')
    item_info_df=item_info_df.rename(columns={'article_id':'click_article_id'})
    return item_info_df

In [40]:
def get_item_emb_dict():
    '''读取文章的embedding'''
    item_emb_df=pd.read_csv('./data/articles_emb.csv')
    item_emb_cols=[x for x in item_emb_df.columns if 'emb' in x]
    # 把embedding转换成numpy数组，ascontiguousarray确保在内存中连续
    item_emb_np=np.ascontiguousarray(item_emb_df[item_emb_cols])
    # L2归一化
    item_emb_np=item_emb_np/np.linalg.norm(item_emb_np,axis=1,keepdims=True)
    item_emb_dict=dict(zip(item_emb_df['article_id'],item_emb_np))
    pickle.dump(item_emb_dict,open('./save/item_content_emb.pkl','wb'))
    return item_emb_dict

In [41]:
all_click_df=get_all_click_sample()
# 对时间戳归一化，按照all_click_df[['click_timestamp']]取出的是DataFrame，使用apply会对列进行操作，而all_click_df['click_timestamp']是对每个元素单独操作，取最值只会取到自己
all_click_df['click_timestamp']=all_click_df[['click_timestamp']].apply(lambda x:(x-np.min(x))/(np.max(x)-np.min(x)))

In [42]:
all_click_df.head()

,user_id,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
56,199982,156624,0.000000,4,4,20,1,21,1
57,199982,156447,0.000141,4,4,20,1,21,1
58,199982,149623,0.000158,4,4,20,1,21,1
178,199940,199198,0.000823,4,1,17,1,25,2
179,199940,272143,0.000831,4,1,17,1,25,2


In [43]:
item_info_df=get_item_info_df()

In [44]:
item_emb_dict=get_item_emb_dict()

# 2.工具函数

In [45]:
def get_user_item_time(click_df):
    '''获取用户-文章-点击时间字典 {user1:{item1:time1,item2:time2}}'''
    memo={}
    for user_id,group in click_df.groupby('user_id'):
        group_sorted=group.sort_values('click_timestamp')
        item_time_dict=dict(zip(group_sorted['click_article_id'],group_sorted['click_timestamp']))
        memo[user_id]=item_time_dict
    return memo

In [46]:
def get_item_user_time(click_df):
    '''获取文章-用户-点击时间字典  {item1: {user1: time1, user2: time2...}...}'''
    memo={}
    for click_article_id,group in click_df.groupby('click_article_id'):
        group_sorted=group.sort_values('click_timestamp')
        item_time_dict=dict(zip(group_sorted['user_id'],group_sorted['click_timestamp']))
        memo[click_article_id]=item_time_dict
    return memo

In [47]:
def get_hist_and_last_click(all_click):
    '''获取当前数据的历史点击（不包括最后一次）和最后一次点击，用于对召回结果检验'''
    all_click = all_click.sort_values(by=['user_id', 'click_timestamp'])
    click_last_df = all_click.groupby('user_id').tail(1)

    # 如果用户只有一个点击，hist为空了，会导致训练的时候这个用户不可见，此时默认泄露一下
    def hist_func(user_df):
        if len(user_df) == 1:
            return user_df
        else:
            return user_df[:-1]

    click_hist_df = all_click.groupby('user_id').apply(hist_func).reset_index(drop=True)

    return click_hist_df, click_last_df

In [48]:
def get_item_info_dict(item_info_df):
    '''获取文章id对应的各个属性'''
    max_min_scaler=lambda x:(x-np.min(x))/(np.max(x)-np.min(x))
    item_info_df['created_at_ts']= item_info_df[['created_at_ts']].apply(max_min_scaler)
    item_type_dict=dict(zip(item_info_df['click_article_id'],item_info_df['category_id']))
    item_words_dict=dict(zip(item_info_df['click_article_id'],item_info_df['words_count']))
    item_created_time_dict=dict(zip(item_info_df['click_article_id'],item_info_df['created_at_ts']))
    return item_type_dict,item_words_dict,item_created_time_dict

In [49]:
def get_item_topk_click(click_df,k):
    '''获取点击次数最多的k个物品'''
    return click_df['click_article_id'].value_counts().index[:k]

In [50]:
def get_user_hist_item_info_dict(all_click):
    '''获取用户历史点击的文章信息'''

    # 获取用户user_id对应的历史点击文章类型的集合字典
    user_hist_item_types=all_click.groupby('user_id')['category_id'].agg(set).reset_index()
    user_hist_item_types_dict=dict(zip(user_hist_item_types['user_id'],user_hist_item_types['category_id']))
    # 获取user_id对应的用户点击文章的集合
    user_hist_item_ids_dict=all_click.groupby('user_id')['click_article_id'].agg(set).reset_index()
    user_hist_item_ids_dict=dict(zip(user_hist_item_ids_dict['user_id'],user_hist_item_ids_dict['click_article_id']))

    # 获取user_id对应的用户历史点击的文章的平均字数字典
    user_hist_item_words=all_click.groupby('user_id')['words_count'].agg("mean").reset_index()
    user_hist_item_words_dict=dict(zip(user_hist_item_words['user_id'],user_hist_item_words['words_count']))

    # 获取user_id对应的用户最后一次点击的文章的创建时间
    all_click_=all_click.sort_values('click_timestamp')
    user_last_item_created_time=all_click_.groupby('user_id')['created_at_ts'].apply(lambda x:x.iloc[-1]).reset_index()
    max_min_scaler = lambda x : (x-np.min(x))/(np.max(x)-np.min(x))
    user_last_item_created_time['created_at_ts']=user_last_item_created_time[['created_at_ts']].apply(max_min_scaler)
    user_last_item_created_time_dict=dict(zip(user_last_item_created_time['user_id'],user_last_item_created_time['created_at_ts']))

    return user_hist_item_types_dict,user_hist_item_ids_dict,user_hist_item_words_dict,user_last_item_created_time_dict

In [51]:
item_type_dict,item_words_dict,item_created_time_dict=get_item_info_dict(item_info_df)

In [52]:
# 定义一个多路召回的字典，将各路召回的结果都保存在这个字典当中
user_multi_recall_dict={'itemcf_sim_itemcf_recall':{},'embedding_sim_item_recall':{},'youtubednn_recall':{},'youtubednn_usercf_recall':{},'cold_start_recall':{}}

In [53]:
trn_hist_click_df,trn_last_click_df=get_hist_and_last_click(all_click_df)

In [54]:
trn_last_click_df

,user_id,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
1112362,37,87694,0.640010,4,3,2,1,25,2
1112271,46,205824,0.639012,4,3,20,1,25,2
1112004,69,205824,0.638756,4,1,12,1,25,2
1111873,87,50644,0.638296,4,3,2,1,25,2
1111863,88,277107,0.638328,4,3,2,1,25,2
...,...,...,...,...,...,...,...,...,...
731020,199885,277492,0.380588,4,1,17,1,13,2
887594,199919,271262,0.473969,4,1,17,1,8,2
907921,199932,236566,0.480282,4,3,20,1,21,2
1076599,199940,336254,0.620134,4,1,17,1,25,2


In [55]:
def metrics_recall(user_recall_items_dict,trn_last_click_df,topk=50):
    # 对召回的结果评估
    last_click_item_dict=dict(zip(trn_last_click_df['user_id'],trn_last_click_df['click_article_id']))
    user_num=len(user_recall_items_dict)
    for k in range(10,topk+1,10):
        hit_num=0
        for user ,item_list in user_recall_items_dict.items():
            tmp=[x[0] for x in user_recall_items_dict[user][:k]]
            if last_click_item_dict[user] in set(tmp):
                hit_num+=1
        hit_rate=round(hit_num*1.0/user_num,5)
        print(' topk: ', k, ' : ', 'hit_num: ', hit_num, 'hit_rate: ', hit_rate, 'user_num : ', user_num)

# 3.计算相似度矩阵

In [56]:
def itemcf_sim(df,item_created_time_dict):
    '''计算物品的相似度矩阵,使用关联规则考虑了1. 用户点击的时间权重 2. 用户点击的顺序权重 3. 文章创建的时间权重'''
    user_item_time_dict =get_user_item_time(df)
    # i2i_sim[i][j]统计i和j共有的受众个数
    i2i_sim=defaultdict(dict)
    # 统计每个物品受众个数
    item_cnt=defaultdict(int)
    for user,item_time_list in tqdm_notebook(user_item_time_dict.items()):
        for loc1,(i,i_click_time) in enumerate(item_time_list.items()):
            # 更新
            item_cnt[i]+=1
            for loc2,(j, j_click_time) in enumerate(item_time_list.items()):
                if i!=j:
                    # 考虑文章的正向顺序点击和反向顺序点击
                    loc_alpha=1.0 if loc2>loc1 else 0.7
                    # 位置信息的权重
                    loc_weight=loc_alpha*(0.9**(np.abs(loc2-loc1)-1))
                    # 点击时间的权重
                    click_time_weight=np.exp(0.7**np.abs(i_click_time-j_click_time))
                    # 创建时间的权重
                    created_time_weight=np.exp(0.8**np.abs(item_created_time_dict[i]-item_created_time_dict[j]))
                    i2i_sim[i].setdefault(j,0)
                    # 加权弱化：用户点击的物品越多，对每对物品的贡献就越小
                    i2i_sim[i][j]+=loc_weight*click_time_weight*created_time_weight/math.log(len(item_time_list)+1)/math.log(len(item_time_list)+1)

    i2i_sim_ = defaultdict(dict)
    for i, related_items in i2i_sim.items():
        for j, wij in related_items.items():
            i2i_sim_[i][j] = wij / math.sqrt(item_cnt[i] * item_cnt[j])


    for i, related_items in i2i_sim.items():
        for j, wij in related_items.items():
            i2i_sim_[i][j]=wij/math.sqrt(item_cnt[i]*item_cnt[j])
    # 保存相似度矩阵
    pickle.dump(i2i_sim_, open('./save/itemcf_i2i_sim.pkl', 'wb'))
    return i2i_sim_

In [57]:
i2i_sim=itemcf_sim(all_click_df,item_created_time_dict)

  0%|          | 0/10000 [00:00<?, ?it/s]

In [59]:
all_click_df.groupby('user_id')['click_article_id'].count()

user_id
37         2
46         2
69         3
87         2
88         2
          ..
199885    13
199919    10
199932     8
199940    16
199982    23
Name: click_article_id, Length: 10000, dtype: int64

In [60]:
def get_user_activate_degree_dict(all_click_df):
    '''获取用户活跃度'''
    all_click_df_=all_click_df.groupby('user_id')['click_article_id'].count().reset_index()

    # 归一化
    mm=MinMaxScaler()
    all_click_df_['click_article_id']=mm.fit_transform(all_click_df_[['click_article_id']])

    user_activate_degree_dict = dict(zip(all_click_df_['user_id'],all_click_df_['click_article_id']))

    return user_activate_degree_dict

In [61]:
def usercf_sim(all_click_df,user_activate_degree_dict):
    '''用户相似度计算'''
    item_user_time_dict=get_item_user_time(all_click_df)
    u2u_sim=defaultdict(dict)
    user_cnt=defaultdict(int)
    for item,user_time_list in tqdm_notebook(item_user_time_dict.items()):
        for u,click_time in user_time_list.items():
            user_cnt[u]+=1
            for v,click_time in user_time_list.items():
                u2u_sim[u].setdefault(v,0)
                if u!=v:
                    activate_weight=100*0.5*(user_activate_degree_dict[u]+user_activate_degree_dict[v])
                    u2u_sim[u][v]+=activate_weight/math.log(len(user_time_list)+1)
    u2u_sim_ = copy.deepcopy(u2u_sim)

    for u,relater_users in u2u_sim.items():
        for v,wij in relater_users.items():
            u2u_sim_[u][v]=wij/math.sqrt(user_cnt[u]*user_cnt[v])
    pickle.dump(u2u_sim_,open('./save/usercf_u2u_sim.pkl', 'wb'))
    return u2u_sim_

In [62]:
user_activate_degree_dict=get_user_activate_degree_dict(all_click_df)
u2u_sim=usercf_sim(all_click_df,user_activate_degree_dict)

  0%|          | 0/6628 [00:00<?, ?it/s]

In [63]:
def embdding_sim(click_df,item_emb_df,topk):
    '''基于文章的embedding计算相似度'''

    # 文章索引与文章id的字典映射
    item_idx_2_rawid_dict=dict(zip(item_emb_df.index,item_emb_df['article_id']))

    item_emb_cols=[x for x in item_emb_df.columns if 'emb' in x]
    item_emb_np=np.ascontiguousarray(item_emb_df[item_emb_cols].values,dtype=np.float32)
    item_emb_np=item_emb_np/np.linalg.norm(item_emb_np,axis=1,keepdims=True)

    # 建立faiss索引，基于内积
    item_index=faiss.IndexFlatIP(item_emb_np.shape[1])
    # 添加向量
    item_index.add(item_emb_np)
    # 为所有物品做一次批量检索，哈走出最相似的k个物品
    sim,idx=item_index.search(item_emb_np,topk)

    item_sim_dict=defaultdict(dict)
    for target_index,sim_value_list,rele_idx_list in tqdm_notebook(zip(range(len(item_emb_np)),sim,idx),total=len(item_emb_np)):
        # sim_value_list,rele_idx_list对当前物品所对的最相似的物品及其相似度
        target_raw_id=item_idx_2_rawid_dict[target_index]
        # 首位是物品本身
        for rele_idx,sim_value in zip(rele_idx_list[1:],sim_value_list[1:]):
            rele_raw_id=item_idx_2_rawid_dict[rele_idx]
            item_sim_dict[target_raw_id][rele_raw_id]=item_sim_dict.get(target_raw_id,{}).get(rele_raw_id,0)+sim_value

    pickle.dump(item_sim_dict,open('./save/emb_i2i_sim.pkl','wb'))
    return item_sim_dict

In [64]:
item_emb_df=pd.read_csv('./data/articles_emb.csv')
emb_i2i_sim=embdding_sim(all_click_df,item_emb_df,topk=10)

  0%|          | 0/364047 [00:00<?, ?it/s]

# 4.召回

## youtubeDnn召回

In [65]:
def gen_data_set(data,negsample=0):
    '''获取youtubeDnn召回召回时的训练和验证数据，这里进行负采样'''
    data.sort_values('click_timestamp',inplace=True)
    item_ids=data['click_article_id'].unique()

    # 数据的格式 [user_id, 交互过的历史物品, 下一个要交互的物品, 正/负本, 历史行为的长度]
    train_set=[]
    test_set=[]

    for reviewerID, hist in tqdm_notebook(data.groupby('user_id')):
        pos_list=hist['click_article_id'].tolist()

        # 用户交互过的物品只有一个也要放到训练集中，否则会造成embedding缺失
        if len(pos_list)==1:
            train_set.append((reviewerID,[pos_list[0]],pos_list[0],1,len(pos_list)))
            test_set.append((reviewerID,[pos_list[0]],pos_list[0],1,len(pos_list)))

        if negsample>0:
            candidate_set=list(set(item_ids)-set(pos_list))
            neg_list=np.random.choice(candidate_set,size=len(pos_list)*negsample,replace=False)# 为每个正样本抽取n个负样本，replace控制是否可以放回抽样
        for i in range(1,len(pos_list)):
            hist=pos_list[:i] # 历史物品
            if i!=len(pos_list)-1:
                train_set.append((reviewerID,hist[::-1],pos_list[i],1,len(hist)))
                for negi in range(negsample):
                    train_set.append((reviewerID,hist[::-1],neg_list[i*negsample+negi],0,len(hist)))
            else:
                # 最长的序列作为测试数据
                test_set.append((reviewerID,hist[::-1],pos_list[i],1,len(hist)))
    random.shuffle(train_set)
    random.shuffle(test_set)
    return train_set,test_set
from tensorflow.keras.preprocessing.sequence import pad_sequences

def gen_model_input(train_set,seq_max_len):
    '''将数据的数据进行padding，使得序列特征长度一致'''

    train_uid=np.array([line[0] for line in train_set]) # 用户id
    train_seq=[line[1] for line in train_set] # 交互过的物品
    train_iid=np.array([line[2] for line in train_set]) # 下一个交互的物品
    train_label=np.array([line[3] for line in train_set])  # 正负/样本
    train_hist_len=np.array([line[4] for line in train_set]) # 序列长度

    # 填充历史行为， post指定往右边填充and截断右边
    train_seq_pad=pad_sequences(train_seq,maxlen=seq_max_len,padding='post',truncating='post',value=0)
    train_model_input={'user_id':train_uid,'click_article_id':train_iid,'hist_article_id':train_seq_pad,'hist_len':train_hist_len}

    return train_model_input,train_label

In [66]:
def youtubednn_u2i_dict(data,topk=20):
    ''''''

## itemCF recall
召回中使用了关联规则
+ 考虑相似文章与历史点击文章顺序的权重
+ 考虑文章创建时间的权重，也就是考虑相似文章与历史点击文章创建时间差的权重
+ 考虑文章内容相似度权重(使用Embedding计算相似文章相似度，但是这里需要注意，在Embedding的时候并没有计算所有商品两两之间的相似度，所以相似的文章与历史点击文章不存在相似度，需要做特殊处理)


In [83]:
def item_based_recommend(user_id,user_item_time_dict,i2i_sim,sim_item_topk,recall_item_num,item_topk_click,item_created_time_dict,emb_i2i_sim):
    """
    基于文章的协同过滤召回
    :param user_id: 用户id
    :param user_item_time_dict: 字典  {user1: {item1: time1, item2: time2..}...}按照时间排序
    :param i2i_sim: 物品相似度矩阵
    :param sim_item_topk: 选择与物品最相似的钱k篇文章
    :param recall_item_num: 最后召回的数量
    :param item_topk_click: 点击次数最多的文章列表，永不补全召回
    :param item_created_time_dict:
    :param emb_i2i_sim: 基于embedding的物品相似度矩阵
    :return:召回的文章列表 {item1:score1, item2: score2...}
    """
    # 获取用户历史交互的文章
    user_hist_items=user_item_time_dict[user_id]

    item_rank={}
    for loc ,(i,click_time) in enumerate(user_hist_items.items()):
        for j,wij in sorted(i2i_sim[i].items(),key=lambda x:x[1],reverse=True)[:sim_item_topk]:
            if j not in user_hist_items:
                # 文章创建时间差权重，时间差越大权重越小，用户更倾向于发布时间按相近的文章
                created_time_weight=np.exp(0.8**np.abs(item_created_time_dict[i]-item_created_time_dict[j]))
                # 位置权重 靠后夫人文章更能代表用户最近的兴趣
                loc_weight=(0.9**(len(user_hist_items)-loc))

                content_weight=1.0
                # 两篇文章越相似权重越大
                content_weight += emb_i2i_sim.get(i, {}).get(j, 0)
                content_weight += emb_i2i_sim.get(j, {}).get(i, 0)

                item_rank.setdefault(j,0)
                item_rank[j]+=created_time_weight*loc_weight*content_weight*wij
    if len(item_rank)<recall_item_num:
        for i,item in enumerate(item_topk_click):
            if item not in item_rank:# 要填充的不在原来的列表中
                item_rank[item]=-i-100 # 随机负数
                if len(item_rank)==recall_item_num:break
    item_rank=sorted(item_rank.items(),key=lambda x:x[1],reverse=True)[:recall_item_num]
    return item_rank

itemCF sim召回

In [79]:
# 判断是否需要检验
if metric_recall:
    trn_hist_click_df,trn_last_click_df=get_hist_and_last_click(all_click_df)
else:
    trn_hist_click_df=all_click_df

# 存储结果的字典
user_recall_items_dict=defaultdict(dict)
#  {user1: {item1: time1, item2: time2..}...}
user_item_time_dict=get_user_item_time(trn_hist_click_df)

i2i_sim=pickle.load(open('./save/itemcf_i2i_sim.pkl','rb'))
emb_i2i_sim=pickle.load(open('./save/emb_i2i_sim.pkl','rb'))

sim_item_topk=20
recall_item_num=10
# 点击次数最多的物品
item_topk_click=get_item_topk_click(trn_hist_click_df,k=50)

for user in tqdm_notebook(trn_hist_click_df['user_id'].unique(),):
    user_recall_items_dict[user]=item_based_recommend(user,user_item_time_dict, i2i_sim, sim_item_topk, recall_item_num, item_topk_click, item_created_time_dict, emb_i2i_sim)

# 存储itemCF sim召回的结果
user_multi_recall_dict['itemcf_sim_itemcf_recall']=user_recall_items_dict
pickle.dump(user_multi_recall_dict['itemcf_sim_itemcf_recall'],open('./save/itemcf_recall_dict.pkl','wb'))

if metric_recall:
    metrics_recall(user_multi_recall_dict['itemcf_sim_itemcf_recall'],trn_last_click_df,topk=recall_item_num)

  0%|          | 0/10000 [00:00<?, ?it/s]

 topk:  10  :  hit_num:  6396 hit_rate:  0.6396 user_num :  10000


embedding sim 召回

In [80]:
if metric_recall:
    trn_hist_click_df,trn_last_click_df=get_hist_and_last_click(all_click_df)
else:
    trn_hist_click_df=all_click_df
user_recall_items_dict=defaultdict(dict)
user_item_time_dict=get_user_item_time(trn_hist_click_df)

# 改一下相似度矩阵即可
i2i_sim=pickle.load(open('./save/emb_i2i_sim.pkl','rb'))

sim_item_topk=20
recall_item_num=10
item_topk_click=get_item_topk_click(trn_hist_click_df,k=50)

for user in tqdm_notebook(trn_hist_click_df['user_id'].unique(),):
    user_recall_items_dict[user]=item_based_recommend(user,user_item_time_dict, i2i_sim, sim_item_topk, recall_item_num, item_topk_click, item_created_time_dict, emb_i2i_sim)

user_multi_recall_dict['embedding_sim_item_recall']=user_recall_items_dict
pickle.dump(user_multi_recall_dict['embedding_sim_item_recall'],open('./save/embedding_sim_item_recall.pkl','wb'))

if metric_recall:
    metrics_recall(user_multi_recall_dict['embedding_sim_item_recall'],trn_last_click_df,topk=recall_item_num)

  0%|          | 0/10000 [00:00<?, ?it/s]

 topk:  10  :  hit_num:  198 hit_rate:  0.0198 user_num :  10000


userCF召回

In [91]:
def user_based_recommand(user_id,user_item_time_dict,u2u_sim,sim_user_topk,recall_item_num,item_topk_click,item_created_time_dict,emb_i2i_sim):
    """
    基于用户的召回u2u2i
    :param user_id: 用户id
    :param user_item_time_dict:  {user1: {item1: time1, item2: time2..}...}
    :param u2u_sim: 用户相似度矩阵
    :param sim_user_topk: 选择与当前用户最相似的前k个用户
    :param recall_item_num: 最后的召回文章数量
    :param item_topk_click: 点击次数最多的文章列表，用户召回补全
    :param item_created_time_dict: 文章创建时间列表
    :param emb_i2i_sim:内容embedding的相似矩阵
    :return: 召回的文章列表 {item1:score1, item2: score2...}
    """

    # 历史交互
    user_item_time_list=user_item_time_dict[user_id]
    user_hist_items=set([i for i,_ in user_item_time_list.items()])

    items_rank={}
    # 相似用户
    for sim_u,wuv in sorted(u2u_sim[user_id].items(),key=lambda x:x[1],reverse=True)[:sim_user_topk]:
        # 相似用户交互过的物品
        for i,click_time in user_item_time_dict[sim_u].items():
            if i not in user_hist_items:
                items_rank.setdefault(i,0)

                loc_weight=1.0
                content_weight=1.0
                created_time_weight=1.0
                # 该物品与当前用户的历史物品做权重交互
                for loc,(j,click_time) in enumerate(user_item_time_list.items()):
                    loc_weight+=0.9**(len(user_item_time_list)-loc)
                    content_weight+=emb_i2i_sim.get(i,{}).get(j,0)
                    content_weight+=emb_i2i_sim.get(j,{}).get(i,0)
                    created_time_weight+=np.exp(0.8*np.abs(item_created_time_dict[i]-item_created_time_dict[j]))

                items_rank[i]+=loc_weight*content_weight*created_time_weight*wuv
    if len(items_rank)<recall_item_num:
        for i,item in enumerate(item_topk_click):
            if item not in items_rank:
                items_rank[item]=-i-100
                if len(items_rank)==recall_item_num:break
    items_rank=sorted(items_rank.items(),key=lambda x:x[1],reverse=True)[:recall_item_num]

    return items_rank


In [92]:
if metric_recall:
    trn_hist_click_df,trn_last_click_df=get_hist_and_last_click(all_click_df)
else:
    trn_hist_click_df=all_click_df
user_recall_items_dict=defaultdict(dict)
user_item_time_dict=get_user_item_time(trn_hist_click_df)

# 改一下相似度矩阵即可
u2u_sim=pickle.load(open('./save/usercf_u2u_sim.pkl','rb'))

sim_item_topk=20
recall_item_num=10
item_topk_click=get_item_topk_click(trn_hist_click_df,k=50)

for user in tqdm_notebook(trn_hist_click_df['user_id'].unique(),):
    user_recall_items_dict[user]=user_based_recommand(user,user_item_time_dict, u2u_sim, sim_item_topk, recall_item_num, item_topk_click, item_created_time_dict, emb_i2i_sim)

pickle.dump(user_recall_items_dict,open('./save/usercf_u2u2i_recall.pkl','wb'))

if metric_recall:
    metrics_recall(user_recall_items_dict,trn_last_click_df,topk=recall_item_num)

  0%|          | 0/10000 [00:00<?, ?it/s]

 topk:  10  :  hit_num:  6367 hit_rate:  0.6367 user_num :  10000
